# prep_imagenes_opencv

Preprocesamiento para el **CAMBIO 09** (no es un experimento: solo genera datos).

Por cada foto (`train_images/`, `test_images/`) calcula con OpenCV:
- **brillo**: media de la imagen en escala de grises (0-255);
- **nitidez**: varianza del Laplaciano (más alto = más nítida);
- **aspecto**: ancho / alto;
- **saturación**: media del canal S en HSV (0-255).

Después agrega por `PetID`: media (`mean`), máximo (`max`) y valor de la primera foto (`first`, archivo `-1.jpg`).
Guarda `train.parquet` y `test.parquet` (12 columnas, índice `PetID`).

En Kaggle: correr este notebook y agregar su salida como input del 09 con el nombre `lab2-prep-imagenes-opencv`.

In [1]:
import glob, os
import cv2
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

def stats_foto(ruta):
    pet_id, n = os.path.basename(ruta)[:-len(".jpg")].rsplit("-", 1)     # archivo "<PetID>-<n>.jpg"
    img = cv2.imread(ruta)                                                  # BGR; None si no se puede leer
    if img is None:
        return {"PetID": pet_id, "foto": int(n)}
    gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    alto, ancho = gris.shape
    return {"PetID": pet_id, "foto": int(n),
            "brillo": float(gris.mean()),
            "nitidez": float(cv2.Laplacian(gris, cv2.CV_64F).var()),
            "aspecto": ancho / alto,
            "saturacion": float(hsv[:, :, 1].mean())}

VARIABLES = ["brillo", "nitidez", "aspecto", "saturacion"]

def por_mascota(carpeta):
    rutas = glob.glob(os.path.join(carpeta, "*.jpg"))
    fotos = pd.DataFrame(Parallel(n_jobs=-1, batch_size=256)(delayed(stats_foto)(r) for r in rutas))
    print(carpeta, "-> fotos:", len(fotos), "| no se pudieron leer:", fotos["brillo"].isna().sum())
    g = fotos.groupby("PetID")[VARIABLES]
    media = g.mean().add_prefix("img_cv_mean_")
    maximo = g.max().add_prefix("img_cv_max_")
    primera = fotos[fotos["foto"] == 1].set_index("PetID")[VARIABLES].add_prefix("img_cv_first_")
    return media.join(maximo).join(primera)

In [2]:
# Local: guarda con la misma ruta que tendría en Kaggle como input. En Kaggle: en la carpeta de trabajo (salida).
SALIDA = "." if os.path.exists("/kaggle") else "../input/lab2-prep-imagenes-opencv"
os.makedirs(SALIDA, exist_ok=True)
for conjunto in ["train", "test"]:
    df = por_mascota(f"../input/petfinder-adoption-prediction/{conjunto}_images")
    df.to_parquet(os.path.join(SALIDA, f"{conjunto}.parquet"))
    print(conjunto, df.shape)
df.describe().T.round(2)

../input/petfinder-adoption-prediction/train_images -> fotos: 58311 | no se pudieron leer: 0
train (14652, 12)


../input/petfinder-adoption-prediction/test_images -> fotos: 14465 | no se pudieron leer: 0
test (3858, 12)


,count,mean,std,min,25%,50%,75%,max
img_cv_mean_brillo,3858.0,120.84,24.35,22.60,106.52,121.55,135.21,229.67
img_cv_mean_nitidez,3858.0,1045.62,1192.44,13.55,316.58,641.27,1304.77,12066.87
img_cv_mean_aspecto,3858.0,1.08,0.31,0.49,0.75,1.08,1.33,2.62
img_cv_mean_saturacion,3858.0,62.07,27.28,0.00,43.39,57.33,76.91,230.23
img_cv_max_brillo,3858.0,135.27,27.39,29.01,118.82,135.24,152.76,239.33
img_cv_max_nitidez,3858.0,1654.45,1949.76,16.22,454.93,961.98,2041.55,17184.13
img_cv_max_aspecto,3858.0,1.22,0.37,0.49,0.79,1.33,1.34,3.81
img_cv_max_saturacion,3858.0,77.70,36.38,0.00,50.87,72.32,98.94,233.65
img_cv_first_brillo,3858.0,121.58,28.68,28.00,103.42,122.31,139.86,229.67
img_cv_first_nitidez,3858.0,1020.86,1348.56,9.09,247.72,555.02,1228.35,13862.66
